In [ ]:
import csv
import re
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import shutil
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from torch.optim import AdamW


TRAIN_PATH     = Path("/content/drive/MyDrive/dontpatronizeme_pcl_cleaned.tsv")
TRAIN_IDS_PATH = Path("/content/drive/MyDrive/train_semeval_parids-labels.csv")
DEV_IDS_PATH   = Path("/content/drive/MyDrive/dev_semeval_parids-labels.csv")

MODEL_NAME       = "cardiffnlp/twitter-roberta-base-sentiment-latest"
MAX_LENGTH       = 200
BATCH_SIZE       = 16
EPOCHS           = 5
LEARNING_RATE    = 1e-5
SEED             = 42
SAVE_PATH        = Path("best_model")
PCL_CLASS_WEIGHT = 4.0

THRESHOLD_RANGE  = np.arange(0.05, 0.95, 0.01)

if Path("best_model").exists():
    shutil.rmtree("best_model")
    print("Deleted old best_model checkpoint")

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def load_parids(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    id_col = "par_id" if "par_id" in df.columns else df.columns[0]
    return pd.DataFrame({"par_id": df[id_col].astype(int)})

def load_clean_tsv(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
        for r in reader:
            if len(r) == 6:
                rows.append(r)

    df = pd.DataFrame(rows, columns=["id", "article_id", "keyword",
                                      "country", "text", "label"])
    df["id"]    = pd.to_numeric(df["id"],    errors="coerce")
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["id", "label"]).reset_index(drop=True)
    df["id"]    = df["id"].astype(int)
    df["label"] = df["label"].astype(int)
    return df

def build_split(text_df: pd.DataFrame, ids_df: pd.DataFrame, name: str) -> pd.DataFrame:
    merged = ids_df.merge(text_df, left_on="par_id", right_on="id", how="inner")
    merged = merged.reset_index(drop=True)
    merged["binary_label"] = (merged["label"].astype(int) >= 2).astype(int)
    print(f"  {name}: {len(merged)} rows  "
          f"(positives: {merged['binary_label'].sum()})")
    return merged

_word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+")

def token_len(text: str) -> int:
    return len(_word_re.findall(str(text).lower()))

def filter_by_length(df: pd.DataFrame, max_len: int = MAX_LENGTH) -> pd.DataFrame:
    lengths = df["text"].apply(token_len)
    before  = len(df)
    df = df[(lengths > 0) & (lengths <= max_len)].reset_index(drop=True)
    print(f"  Length filter: {before} -> {len(df)} rows  "
          f"(removed {before - len(df)})")
    return df

class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        print("  Tokenising...", flush=True)
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
        print(f"  Done. {len(labels)} examples tokenised.", flush=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

def train_epoch(model, loader, optimizer, scheduler, criterion, epoch, total_epochs):
    model.train()
    total_loss = 0.0
    bar = tqdm(loader, desc=f"  Epoch {epoch}/{total_epochs} [train]", leave=True)
    for batch in bar:
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader)

def diagnose_probs(probs, labels):
    pcl_probs    = probs[labels == 1]
    no_pcl_probs = probs[labels == 0]
    print(f"\n  [Diagnostics]")
    print(f"  PCL    probs — mean: {pcl_probs.mean():.4f}  "
          f"max: {pcl_probs.max():.4f}  "
          f"min: {pcl_probs.min():.4f}")
    print(f"  No PCL probs — mean: {no_pcl_probs.mean():.4f}  "
          f"max: {no_pcl_probs.max():.4f}  "
          f"min: {no_pcl_probs.min():.4f}")
    for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
        above = (pcl_probs >= t).sum()
        print(f"  PCL examples scoring >= {t}: {above}/{len(pcl_probs)}")


def get_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Evaluating", leave=True):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].numpy()

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels)

    return np.array(all_probs), np.array(all_labels)


def tune_threshold(probs, labels, threshold_range=THRESHOLD_RANGE):
    best_t, best_f1 = 0.5, 0.0
    for t in threshold_range:
        preds = (probs >= t).astype(int)
        f1    = f1_score(labels, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t  = round(float(t), 4)
    print(f"  Best threshold: {best_t:.2f}  |  Dev F1 (PCL): {best_f1:.4f}")
    return best_t, best_f1


def evaluate(model, loader, threshold=0.5, label="Eval"):
    probs, labels = get_probs(model, loader)
    preds = (probs >= threshold).astype(int)
    f1 = f1_score(labels,   preds, pos_label=1, zero_division=0)
    p  = precision_score(labels, preds, pos_label=1, zero_division=0)
    r  = recall_score(labels,    preds, pos_label=1, zero_division=0)
    print(f"\n[{label}]  threshold={threshold:.2f}  "
          f"P={p:.4f}  R={r:.4f}  F1={f1:.4f}")
    print(classification_report(labels, preds,
                                 target_names=["No PCL", "PCL"],
                                 zero_division=0))
    diagnose_probs(probs, labels)
    return f1, probs, labels

def main():
    print("Loading par_id split files...")
    train_ids = load_parids(TRAIN_IDS_PATH)
    dev_ids   = load_parids(DEV_IDS_PATH)
    print(f"  Train IDs: {len(train_ids)}")
    print(f"  Dev   IDs: {len(dev_ids)}")

    print("\nLoading cleaned text TSV...")
    text_df = load_clean_tsv(TRAIN_PATH)
    print(f"  Loaded {len(text_df)} text rows")

    print("\nBuilding splits...")
    train_df = build_split(text_df, train_ids, "Train")
    dev_df   = build_split(text_df, dev_ids,   "Dev  ")

    print("\nApplying length filter to train...")
    train_df = filter_by_length(train_df, max_len=MAX_LENGTH)
    dev_df   = dev_df[dev_df["text"].apply(token_len) > 0].reset_index(drop=True)

    print(f"\nFinal train: {len(train_df)} rows  "
          f"| positives: {train_df['binary_label'].sum()}")
    print(f"Final dev:   {len(dev_df)} rows  "
          f"| positives: {dev_df['binary_label'].sum()}")
    print(f"\nLabel counts (train):\n"
          f"{train_df['binary_label'].value_counts().to_string()}")

    print(f"\nLoading tokeniser for {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print("\nTokenising train set...")
    train_dataset = PCLDataset(
        train_df["text"].tolist(),
        train_df["binary_label"].tolist(),
        tokenizer,
    )
    print("Tokenising dev set...")
    dev_dataset = PCLDataset(
        dev_df["text"].tolist(),
        dev_df["binary_label"].tolist(),
        tokenizer,
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

    print(f"\nLoading {MODEL_NAME}...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        ignore_mismatched_sizes=True,
    ).to(device)

    class_weights = torch.tensor([1.0, PCL_CLASS_WEIGHT],
                                  dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f"\nClass weights: No PCL=1.0  PCL={PCL_CLASS_WEIGHT}")

    optimizer    = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = len(train_loader)
    print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_dev_f1    = 0.0
    best_threshold = 0.5

    print("\n" + "=" * 60)
    print(f"Training {MODEL_NAME} with class-weighted loss")
    print("=" * 60)

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(
            model, train_loader, optimizer, scheduler,
            criterion, epoch, EPOCHS,
        )
        print(f"\nEpoch {epoch}/{EPOCHS}  |  Train loss: {train_loss:.4f}")

        _, dev_probs, dev_labels = evaluate(
            model, dev_loader,
            threshold=0.5,
            label=f"Dev epoch {epoch} (t=0.5)",
        )

        best_t, tuned_f1 = tune_threshold(dev_probs, dev_labels)

        if tuned_f1 > best_dev_f1:
            best_dev_f1    = tuned_f1
            best_threshold = best_t
            SAVE_PATH.mkdir(exist_ok=True)
            model.save_pretrained(SAVE_PATH)
            tokenizer.save_pretrained(SAVE_PATH)
            print(f"  --> Best model saved  "
                  f"(F1={best_dev_f1:.4f}, threshold={best_threshold:.2f})")

    print("\n" + "=" * 60)
    print(f"Training complete.")
    print(f"Best dev positive-class F1: {best_dev_f1:.4f}  "
          f"at threshold {best_threshold:.2f}")

    best_model = AutoModelForSequenceClassification.from_pretrained(
        SAVE_PATH).to(device)
    evaluate(
        best_model, dev_loader,
        threshold=best_threshold,
        label="Final dev (best model + tuned threshold)",
    )

    (SAVE_PATH / "threshold.txt").write_text(str(best_threshold))
    print(f"\nThreshold saved to: {SAVE_PATH}/threshold.txt")
    print(f"Model saved to:     {SAVE_PATH.resolve()}/")


if __name__ == "__main__":
    main()

Using device: cuda
Loading par_id split files...
  Train IDs: 8375
  Dev   IDs: 2094

Loading cleaned text TSV...
  Loaded 10469 text rows

Building splits...
  Train: 8375 rows  (positives: 794)
  Dev  : 2094 rows  (positives: 199)

Applying length filter to train...
  Length filter: 8375 -> 8368 rows  (removed 7)

Final train: 8368 rows  | positives: 792
Final dev:   2093 rows  | positives: 199

Label counts (train):
binary_label
0    7576
1     792

Loading tokeniser for cardiffnlp/twitter-roberta-base-sentiment-latest...

Tokenising train set...
  Tokenising...
  Done. 8368 examples tokenised.
Tokenising dev set...
  Tokenising...
  Done. 2093 examples tokenised.

Loading cardiffnlp/twitter-roberta-base-sentiment-latest...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:to


Class weights: No PCL=1.0  PCL=4.0
Total steps: 2615  |  Warmup steps: 523

Training cardiffnlp/twitter-roberta-base-sentiment-latest with class-weighted loss


  Epoch 1/5 [train]: 100%|██████████| 523/523 [04:48<00:00,  1.81it/s, loss=0.6834]


...

  Evaluating: 100%|██████████| 131/131 [00:21<00:00,  6.08it/s]



[Dev epoch 5 (t=0.5)]  threshold=0.50  P=0.6127  R=0.5327  F1=0.5699
              precision    recall  f1-score   support

      No PCL       0.95      0.96      0.96      1894
         PCL       0.61      0.53      0.57       199

    accuracy                           0.92      2093
   macro avg       0.78      0.75      0.76      2093
weighted avg       0.92      0.92      0.92      2093


  [Diagnostics]
  PCL    probs — mean: 0.5245  max: 0.9979  min: 0.0005
  No PCL probs — mean: 0.0385  max: 0.9980  min: 0.0004
  PCL examples scoring >= 0.1: 116/199
  PCL examples scoring >= 0.2: 112/199
  PCL examples scoring >= 0.3: 110/199
  PCL examples scoring >= 0.4: 107/199
  PCL examples scoring >= 0.5: 106/199
  Best threshold: 0.05  |  Dev F1 (PCL): 0.5762

Training complete.
Best dev positive-class F1: 0.5891  at threshold 0.30


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Evaluating: 100%|██████████| 131/131 [00:21<00:00,  6.08it/s]



[Final dev (best model + tuned threshold)]  threshold=0.30  P=0.5586  R=0.6231  F1=0.5891
              precision    recall  f1-score   support

      No PCL       0.96      0.95      0.95      1894
         PCL       0.56      0.62      0.59       199

    accuracy                           0.92      2093
   macro avg       0.76      0.79      0.77      2093
weighted avg       0.92      0.92      0.92      2093


  [Diagnostics]
  PCL    probs — mean: 0.5785  max: 0.9971  min: 0.0009
  No PCL probs — mean: 0.0499  max: 0.9970  min: 0.0007
  PCL examples scoring >= 0.1: 129/199
  PCL examples scoring >= 0.2: 126/199
  PCL examples scoring >= 0.3: 124/199
  PCL examples scoring >= 0.4: 119/199
  PCL examples scoring >= 0.5: 115/199

Threshold saved to: best_model/threshold.txt
Model saved to:     /content/best_model/
